In [2]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
experimento = mlflow.set_experiment(
    "Sistema de alerta temprana de volatilidad para acciones"
)

print("URI activa:", mlflow.get_tracking_uri())
print("Experimento:", experimento.name)

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


URI activa: http://127.0.0.1:5000
Experimento: Sistema de alerta temprana de volatilidad para acciones


In [3]:
import os
import sys
from hashlib import sha256
from pathlib import Path

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent
sys.path.insert(0, str(raiz))

from src.datos import cargar_obt

df = cargar_obt()

snapshot = df.sort_values(["TRADE_DATE", "SYMBOL"])
snapshot_bytes = snapshot.to_csv(
    index=False,
    lineterminator="\n",
).encode("utf-8")

datos_corrida = {
    "tabla_origen": "PSET2_DB.OBT.AAPL_DAILY_OBT",
    "fecha_corte_snapshot": os.environ["DATA_CUTOFF"],
    "sha256_snapshot_obt": sha256(snapshot_bytes).hexdigest(),
    "filas_snapshot": len(snapshot),
}

print("Dimensiones de la OBT:", df.shape)
print(datos_corrida)

Dimensiones de la OBT: (4087, 19)
{'tabla_origen': 'PSET2_DB.OBT.AAPL_DAILY_OBT', 'fecha_corte_snapshot': '2026-09-30', 'sha256_snapshot_obt': '9e9d8a678f5feb51385ae330e460a538835644c6e2068e9f989fd362ce45d4e8', 'filas_snapshot': 4087}


In [4]:
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

from src.features import construir_features

horizonte = 5
inicio_prueba = pd.Timestamp("2023-07-11")

features = construir_features(df)

vol_futura_5 = (
    features["RET_LAG_0"]
    .rolling(horizonte, min_periods=horizonte)
    .std(ddof=1)
    .shift(-horizonte)
    .rename("VOL_FUTURA_5")
)

# Calcular el fin de cada ventana antes de excluir las ruedas iniciales.
fin_ventana_target = features.index.to_series().shift(-horizonte)

datos_modelado = (
    features.iloc[60:]
    .join(vol_futura_5)
    .dropna(subset=["VOL_FUTURA_5"])
)

fin_ventana_modelado = fin_ventana_target.reindex(datos_modelado.index)
entrenamiento = datos_modelado.loc[
    fin_ventana_modelado < inicio_prueba
].copy()

X_desarrollo = entrenamiento.drop(columns="VOL_FUTURA_5")
vol_desarrollo = entrenamiento["VOL_FUTURA_5"]
columnas_numericas = X_desarrollo.columns.drop("MES").tolist()

cv_temporal = TimeSeriesSplit(n_splits=5, gap=horizonte)

print("Dataset supervisado:", datos_modelado.shape)
print("Features de entrenamiento:", X_desarrollo.shape)
print("Última fecha de alerta en entrenamiento:", X_desarrollo.index.max().date())
print("Inicio del hold-out:", inicio_prueba.date())

Dataset supervisado: (4022, 19)
Features de entrenamiento: (3212, 18)
Última fecha de alerta en entrenamiento: 2023-06-30
Inicio del hold-out: 2023-07-11


## Dummy Classifier

In [ ]:
#BASELINE DEL DUMMY CLASSIFIER

from time import perf_counter

from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

from src.pipeline import crear_pipeline

resultados_dummy = []

for pliegue, (idx_train, idx_val) in enumerate(
    cv_temporal.split(X_desarrollo), start=1
):
    X_train = X_desarrollo.iloc[idx_train]
    X_val = X_desarrollo.iloc[idx_val]

    vol_train = vol_desarrollo.iloc[idx_train]
    vol_val = vol_desarrollo.iloc[idx_val]

    p80 = vol_train.quantile(0.80)
    y_train = vol_train.gt(p80).astype(int)
    y_val = vol_val.gt(p80).astype(int)

    dummy = crear_pipeline(
        DummyClassifier(strategy="most_frequent"),
        columnas_numericas,
    )

    inicio = perf_counter()
    dummy.fit(X_train, y_train)
    segundos_ajuste = perf_counter() - inicio

    proba_train = dummy.predict_proba(X_train)[:, 1]
    proba_val = dummy.predict_proba(X_val)[:, 1]

    resultados_dummy.append({
        "pliegue": pliegue,
        "p80_train": p80,
        "positivos_validacion": int(y_val.sum()),
        "roc_auc_train": roc_auc_score(y_train, proba_train),
        "roc_auc_val": roc_auc_score(y_val, proba_val),
        "segundos_ajuste": segundos_ajuste,
    })

resultados_dummy = pd.DataFrame(resultados_dummy)
display(resultados_dummy.round(4))

,pliegue,p80_train,positivos_validacion,roc_auc_train,roc_auc_val,segundos_ajuste
0,1,1.8833,106,0.5,0.5,0.0118
1,2,1.8898,110,0.5,0.5,0.0510
2,3,1.8949,105,0.5,0.5,0.0300
3,4,1.8921,218,0.5,0.5,0.0403
4,5,2.0642,133,0.5,0.5,0.0246


In [5]:
import subprocess
from time import perf_counter

import mlflow.sklearn
from mlflow.models import infer_signature

commit_git = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=raiz, text=True
).strip()

cambios_sin_commit = bool(subprocess.check_output(
    ["git", "status", "--porcelain", "-uno"],
    cwd=raiz,
    text=True,
).strip())

hash_desarrollo = sha256(
    entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
).hexdigest()

p80_desarrollo = vol_desarrollo.quantile(0.80)
y_desarrollo = vol_desarrollo.gt(p80_desarrollo).astype(int)

dummy_reajustado = crear_pipeline(
    DummyClassifier(strategy="most_frequent"),
    columnas_numericas,
)

inicio = perf_counter()
dummy_reajustado.fit(X_desarrollo, y_desarrollo)
segundos_reajuste = perf_counter() - inicio

ejemplo = X_desarrollo.head(5)
firma = infer_signature(ejemplo, dummy_reajustado.predict_proba(ejemplo))

auc_train = resultados_dummy["roc_auc_train"].mean()
auc_val = resultados_dummy["roc_auc_val"].mean()

with mlflow.start_run(run_name="baseline_dummy_most_frequent") as corrida:
    mlflow.set_tags({
        "fase": "baseline",
        "familia": "DummyClassifier",
        "git_commit": commit_git,
        "git_cambios_sin_commit": str(cambios_sin_commit).lower(),
    })

    mlflow.log_params({
        **datos_corrida,
        "sha256_dataset_desarrollo": hash_desarrollo,
        "estrategia": "most_frequent",
        "semilla": "no_aplica",
        "n_pliegues": cv_temporal.n_splits,
        "gap": horizonte,
        "inicio_holdout": inicio_prueba.date().isoformat(),
        "p80_desarrollo": float(p80_desarrollo),
    })

    mlflow.log_metrics({
        "roc_auc_train_media_cv": float(auc_train),
        "roc_auc_val_media_cv": float(auc_val),
        "roc_auc_val_std_cv": float(resultados_dummy["roc_auc_val"].std(ddof=1)),
        "brecha_roc_auc_cv": float(auc_train - auc_val),
        "segundos_ajuste_cv_total": float(resultados_dummy["segundos_ajuste"].sum()),
        "segundos_reajuste_desarrollo": segundos_reajuste,
    })

    mlflow.log_text(
        resultados_dummy.to_csv(index=False),
        "resultados_por_pliegue.csv",
    )

    modelo_registrado = mlflow.sklearn.log_model(
        dummy_reajustado,
        name="pipeline_dummy",
        signature=firma,
        input_example=ejemplo,
        pyfunc_predict_fn="predict_proba",
    )

print("Run ID:", corrida.info.run_id)
print("URI del modelo:", modelo_registrado.model_uri)

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/22 15:38:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run baseline_dummy_most_frequent at: http://127.0.0.1:5000/#/experiments/1/runs/10368c341481406098438197c2a16f14
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: 10368c341481406098438197c2a16f14
URI del modelo: models:/m-aa55df3fbb444158b5017418e9903a39


## Baseline Persistencia

In [6]:
#BASELINE "Random Walk" para volatilidad (persistencia)

resultados_persistencia = []

for pliegue, (idx_train, idx_val) in enumerate(
    cv_temporal.split(X_desarrollo), start=1
):
    vol_train = vol_desarrollo.iloc[idx_train]
    vol_val = vol_desarrollo.iloc[idx_val]

    p80 = vol_train.quantile(0.80)
    y_train = vol_train.gt(p80).astype(int)
    y_val = vol_val.gt(p80).astype(int)

    score_train = X_desarrollo.iloc[idx_train]["VOL_HIST_5"]
    score_val = X_desarrollo.iloc[idx_val]["VOL_HIST_5"]
    alertas_val = score_val.gt(p80)

    resultados_persistencia.append({
        "pliegue": pliegue,
        "p80_train": p80,
        "positivos_validacion": int(y_val.sum()),
        "alertas_validacion": int(alertas_val.sum()),
        "roc_auc_train": roc_auc_score(y_train, score_train),
        "roc_auc_val": roc_auc_score(y_val, score_val),
    })

resultados_persistencia = pd.DataFrame(resultados_persistencia)
display(resultados_persistencia.round(4))

auc_cv = resultados_persistencia["roc_auc_val"]
print(f"ROC-AUC de validación: {auc_cv.mean():.4f} ± {auc_cv.std(ddof=1):.4f}")

,pliegue,p80_train,positivos_validacion,alertas_validacion,roc_auc_train,roc_auc_val
0,1,1.8833,106,109,0.5308,0.5771
1,2,1.8898,110,110,0.5613,0.5978
2,3,1.8949,105,105,0.5702,0.7414
3,4,1.8921,218,216,0.6170,0.7113
4,5,2.0642,133,133,0.6707,0.6989


ROC-AUC de validación: 0.6653 ± 0.0731


In [7]:
from hashlib import sha256
import subprocess

commit_git = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=raiz, text=True
).strip()

cambios_sin_commit = bool(subprocess.check_output(
    ["git", "status", "--porcelain", "-uno"], cwd=raiz, text=True
).strip())

hash_desarrollo = sha256(
    entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
).hexdigest()

auc_train = resultados_persistencia["roc_auc_train"].mean()
auc_val = resultados_persistencia["roc_auc_val"].mean()
p80_desarrollo = float(vol_desarrollo.quantile(0.80))

with mlflow.start_run(run_name="baseline_persistencia_vol5") as corrida:
    mlflow.set_tags({
        "fase": "baseline",
        "familia": "persistencia",
        "git_commit": commit_git,
        "git_cambios_sin_commit": str(cambios_sin_commit).lower(),
    })

    mlflow.log_params({
        **datos_corrida,
        "sha256_dataset_desarrollo": hash_desarrollo,
        "estrategia": "volatilidad_historica_como_pronostico",
        "feature_score": "VOL_HIST_5",
        "semilla": "no_aplica",
        "n_pliegues": cv_temporal.n_splits,
        "gap": horizonte,
        "inicio_holdout": inicio_prueba.date().isoformat(),
        "p80_desarrollo": p80_desarrollo,
    })

    mlflow.log_metrics({
        "roc_auc_train_media_cv": float(auc_train),
        "roc_auc_val_media_cv": float(auc_val),
        "roc_auc_val_std_cv": float(
            resultados_persistencia["roc_auc_val"].std(ddof=1)
        ),
        "brecha_roc_auc_cv": float(auc_train - auc_val),
    })

    mlflow.log_text(
        resultados_persistencia.to_csv(index=False),
        "resultados_por_pliegue.csv",
    )

    mlflow.log_dict({
        "score": "VOL_HIST_5",
        "columna_para_estimar_umbral": "VOL_FUTURA_5",
        "cuantil": 0.80,
        "condicion_alerta": "score > umbral",
        "p80_desarrollo": p80_desarrollo,
    }, "regla_persistencia.json")

print("Run ID:", corrida.info.run_id)

🏃 View run baseline_persistencia_vol5 at: http://127.0.0.1:5000/#/experiments/1/runs/fd903f015c3e4ab19d1f767e02af6d7a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: fd903f015c3e4ab19d1f767e02af6d7a


# Baseline de Negocio: Buy and Hold

In [8]:
capital_inicial = 10_000.0

# RET_LAG_0 está en porcentaje. Alinear el próximo retorno con la fecha t.
retorno_siguiente = features["RET_LAG_0"].shift(-1).div(100)

resultados_buy_hold = []

for pliegue, (_, idx_val) in enumerate(
    cv_temporal.split(X_desarrollo), start=1
):
    fechas_val = X_desarrollo.index[idx_val]
    retornos = retorno_siguiente.loc[fechas_val]

    assert retornos.notna().all(), f"Faltan retornos en el pliegue {pliegue}"

    capital = capital_inicial * (1 + retornos).cumprod()
    maximo_acumulado = capital.cummax().clip(lower=capital_inicial)
    drawdown = capital.div(maximo_acumulado).sub(1)

    resultados_buy_hold.append({
        "pliegue": pliegue,
        "inicio_validacion": fechas_val[0].date(),
        "fin_validacion": fechas_val[-1].date(),
        "capital_final_usd": capital.iloc[-1],
        "rendimiento_acumulado_pct": (capital.iloc[-1] / capital_inicial - 1) * 100,
        "max_drawdown_pct": -drawdown.min() * 100,
    })

resultados_buy_hold = pd.DataFrame(resultados_buy_hold)
display(resultados_buy_hold.round(2))

,pliegue,inicio_validacion,fin_validacion,capital_final_usd,rendimiento_acumulado_pct,max_drawdown_pct
0,1,2012-11-13,2014-12-29,15185.46,51.85,33.37
1,2,2014-12-30,2017-02-13,12529.72,25.30,30.44
2,3,2017-02-14,2019-04-01,14819.32,48.19,38.52
3,4,2019-04-02,2021-05-14,26622.68,166.23,31.43
4,5,2021-05-17,2023-06-30,15420.56,54.21,30.91


In [9]:
commit_git = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=raiz, text=True
).strip()

cambios_sin_commit = bool(subprocess.check_output(
    ["git", "status", "--porcelain", "-uno"], cwd=raiz, text=True
).strip())

with mlflow.start_run(run_name="baseline_buy_hold") as corrida:
    mlflow.set_tags({
        "fase": "baseline",
        "familia": "regla_negocio",
        "git_commit": commit_git,
        "git_cambios_sin_commit": str(cambios_sin_commit).lower(),
    })

    mlflow.log_params({
        **datos_corrida,
        "sha256_dataset_desarrollo": hash_desarrollo,
        "estrategia": "buy_and_hold",
        "capital_inicial_usd": capital_inicial,
        "exposicion_aapl": 1.0,
        "costos_transaccion": 0.0,
        "retorno": "cierre_ajustado_t_a_cierre_ajustado_t_mas_1",
        "reinicio_capital_por_pliegue": True,
        "n_pliegues": cv_temporal.n_splits,
        "gap": horizonte,
        "inicio_holdout": inicio_prueba.date().isoformat(),
    })

    mlflow.log_metrics({
        "capital_final_usd_media_cv": float(
            resultados_buy_hold["capital_final_usd"].mean()
        ),
        "max_drawdown_pct_media_cv": float(
            resultados_buy_hold["max_drawdown_pct"].mean()
        ),
    })

    mlflow.log_text(
        resultados_buy_hold.to_csv(index=False),
        "resultados_por_pliegue.csv",
    )

print("Run ID:", corrida.info.run_id)

🏃 View run baseline_buy_hold at: http://127.0.0.1:5000/#/experiments/1/runs/2eca97e8b9224d4c879714ae1668784e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: 2eca97e8b9224d4c879714ae1668784e


## Primer candiato: Regresión Logística con Regularización

In [12]:
from time import perf_counter

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from src.pipeline import crear_pipeline

SEMILLA = 42
config_logistica = {"random_state": SEMILLA}
resultados_logistica = []

for pliegue, (idx_train, idx_val) in enumerate(
    cv_temporal.split(X_desarrollo), start=1
):
    X_train = X_desarrollo.iloc[idx_train]
    X_val = X_desarrollo.iloc[idx_val]

    vol_train = vol_desarrollo.iloc[idx_train]
    vol_val = vol_desarrollo.iloc[idx_val]

    p80 = vol_train.quantile(0.80)
    y_train = vol_train.gt(p80).astype(int)
    y_val = vol_val.gt(p80).astype(int)

    logistica = crear_pipeline(
        LogisticRegression(**config_logistica),
        columnas_numericas,
    )

    inicio = perf_counter()
    logistica.fit(X_train, y_train)
    segundos_ajuste = perf_counter() - inicio

    auc_train = roc_auc_score(y_train, logistica.predict_proba(X_train)[:, 1])
    auc_val = roc_auc_score(y_val, logistica.predict_proba(X_val)[:, 1])

    resultados_logistica.append({
        "pliegue": pliegue,
        "fin_train": X_train.index[-1].date(),
        "inicio_validacion": X_val.index[0].date(),
        "fin_validacion": X_val.index[-1].date(),
        "p80_train": p80,
        "roc_auc_train": auc_train,
        "roc_auc_val": auc_val,
        "brecha_roc_auc": auc_train - auc_val,
        "segundos_ajuste": segundos_ajuste,
    })

resultados_logistica = pd.DataFrame(resultados_logistica)
display(resultados_logistica.round(4))

auc_cv = resultados_logistica["roc_auc_val"]
print(f"ROC-AUC de validación: {auc_cv.mean():.4f} ± {auc_cv.std(ddof=1):.4f}")

,pliegue,fin_train,inicio_validacion,fin_validacion,p80_train,roc_auc_train,roc_auc_val,brecha_roc_auc,segundos_ajuste
0,1,2012-11-05,2012-11-13,2014-12-29,1.8833,0.8075,0.6003,0.2072,0.2328
1,2,2014-12-19,2014-12-30,2017-02-13,1.8898,0.7566,0.7269,0.0297,0.0830
2,3,2017-02-06,2017-02-14,2019-04-01,1.8949,0.7589,0.7797,-0.0207,0.1099
3,4,2019-03-25,2019-04-02,2021-05-14,1.8921,0.7735,0.7441,0.0294,0.0604
4,5,2021-05-07,2021-05-17,2023-06-30,2.0642,0.8087,0.6983,0.1104,0.0649


ROC-AUC de validación: 0.7099 ± 0.0679


In [13]:
import subprocess
from hashlib import sha256

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

commit_git = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=raiz, text=True
).strip()

cambios_sin_commit = bool(subprocess.check_output(
    ["git", "status", "--porcelain", "-uno"], cwd=raiz, text=True
).strip())

hash_desarrollo = sha256(
    entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
).hexdigest()

p80_desarrollo = float(vol_desarrollo.quantile(0.80))
y_desarrollo = vol_desarrollo.gt(p80_desarrollo).astype(int)

logistica_reajustada = crear_pipeline(
    LogisticRegression(**config_logistica),
    columnas_numericas,
)

inicio = perf_counter()
logistica_reajustada.fit(X_desarrollo, y_desarrollo)
segundos_reajuste = perf_counter() - inicio

estimador = logistica_reajustada.named_steps["modelo"]
hiperparametros = {
    f"modelo__{nombre}": valor
    for nombre, valor in estimador.get_params(deep=False).items()
}

ejemplo = X_desarrollo.head(5).copy()
firma = infer_signature(ejemplo, logistica_reajustada.predict_proba(ejemplo))

with mlflow.start_run(run_name="candidato_regresion_logistica_default") as corrida:
    mlflow.set_tags({
        "fase": "candidato",
        "familia": "lineal_regularizado",
        "git_commit": commit_git,
        "git_cambios_sin_commit": str(cambios_sin_commit).lower(),
    })

    mlflow.log_params({
        **datos_corrida,
        **hiperparametros,
        "sha256_dataset_desarrollo": hash_desarrollo,
        "modelo": "LogisticRegression",
        "semilla": SEMILLA,
        "n_pliegues": cv_temporal.n_splits,
        "gap": cv_temporal.gap,
        "inicio_holdout": inicio_prueba.date().isoformat(),
        "p80_desarrollo": p80_desarrollo,
    })

    mlflow.log_metrics({
        "roc_auc_train_media_cv": float(resultados_logistica["roc_auc_train"].mean()),
        "roc_auc_val_media_cv": float(resultados_logistica["roc_auc_val"].mean()),
        "roc_auc_val_std_cv": float(resultados_logistica["roc_auc_val"].std(ddof=1)),
        "brecha_roc_auc_cv": float(resultados_logistica["brecha_roc_auc"].mean()),
        "segundos_ajuste_cv_total": float(resultados_logistica["segundos_ajuste"].sum()),
        "segundos_reajuste_desarrollo": segundos_reajuste,
    })

    mlflow.log_text(
        resultados_logistica.to_csv(index=False),
        "resultados_por_pliegue.csv",
    )

    modelo_logistica = mlflow.sklearn.log_model(
        logistica_reajustada,
        name="pipeline_logistica",
        signature=firma,
        input_example=ejemplo,
        pyfunc_predict_fn="predict_proba",
    )

print("Run ID:", corrida.info.run_id)
print("URI del modelo:", modelo_logistica.model_uri)

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/22 16:12:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run candidato_regresion_logistica_default at: http://127.0.0.1:5000/#/experiments/1/runs/b1d407af9d61477c90cf10b99db6df1a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: b1d407af9d61477c90cf10b99db6df1a
URI del modelo: models:/m-e397471cd01b493e8e8691cd99ef8a7e


## Segundo Candidato: Random Forest

In [15]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier


def evaluar_candidato(estimador):
    resultados = []

    for pliegue, (idx_train, idx_val) in enumerate(
        cv_temporal.split(X_desarrollo), start=1
    ):
        X_train = X_desarrollo.iloc[idx_train]
        X_val = X_desarrollo.iloc[idx_val]
        vol_train = vol_desarrollo.iloc[idx_train]
        vol_val = vol_desarrollo.iloc[idx_val]

        p80 = vol_train.quantile(0.80)
        y_train = vol_train.gt(p80).astype(int)
        y_val = vol_val.gt(p80).astype(int)

        pipeline = crear_pipeline(clone(estimador), columnas_numericas)

        inicio = perf_counter()
        pipeline.fit(X_train, y_train)
        segundos_ajuste = perf_counter() - inicio

        auc_train = roc_auc_score(y_train, pipeline.predict_proba(X_train)[:, 1])
        auc_val = roc_auc_score(y_val, pipeline.predict_proba(X_val)[:, 1])

        resultados.append({
            "pliegue": pliegue,
            "fin_train": X_train.index[-1].date(),
            "inicio_validacion": X_val.index[0].date(),
            "fin_validacion": X_val.index[-1].date(),
            "p80_train": p80,
            "roc_auc_train": auc_train,
            "roc_auc_val": auc_val,
            "brecha_roc_auc": auc_train - auc_val,
            "segundos_ajuste": segundos_ajuste,
        })

    return pd.DataFrame(resultados)


bosque = RandomForestClassifier(random_state=SEMILLA)
resultados_bosque = evaluar_candidato(bosque)

display(resultados_bosque.round(4))

auc_cv = resultados_bosque["roc_auc_val"]
print(f"ROC-AUC de validación: {auc_cv.mean():.4f} ± {auc_cv.std(ddof=1):.4f}")
print(f"Brecha media: {resultados_bosque['brecha_roc_auc'].mean():.4f}")

,pliegue,fin_train,inicio_validacion,fin_validacion,p80_train,roc_auc_train,roc_auc_val,brecha_roc_auc,segundos_ajuste
0,1,2012-11-05,2012-11-13,2014-12-29,1.8833,1.0,0.6476,0.3524,0.7399
1,2,2014-12-19,2014-12-30,2017-02-13,1.8898,1.0,0.7080,0.2920,1.6545
2,3,2017-02-06,2017-02-14,2019-04-01,1.8949,1.0,0.7437,0.2563,1.6061
3,4,2019-03-25,2019-04-02,2021-05-14,1.8921,1.0,0.7323,0.2677,1.5535
4,5,2021-05-07,2021-05-17,2023-06-30,2.0642,1.0,0.7775,0.2225,2.1078


ROC-AUC de validación: 0.7218 ± 0.0484
Brecha media: 0.2782


In [16]:
def registrar_candidato(estimador, resultados, *, nombre_corrida, familia, nombre_pipeline):
    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=raiz, text=True
    ).strip()
    cambios_pendientes = bool(subprocess.check_output(
        ["git", "status", "--porcelain", "-uno"], cwd=raiz, text=True
    ).strip())

    hash_dataset = sha256(
        entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
    ).hexdigest()

    p80_desarrollo = float(vol_desarrollo.quantile(0.80))
    y_desarrollo = vol_desarrollo.gt(p80_desarrollo).astype(int)

    pipeline = crear_pipeline(clone(estimador), columnas_numericas)
    inicio = perf_counter()
    pipeline.fit(X_desarrollo, y_desarrollo)
    segundos_reajuste = perf_counter() - inicio

    parametros = estimador.get_params(deep=False)
    ejemplo = X_desarrollo.head(5).copy()
    firma = infer_signature(ejemplo, pipeline.predict_proba(ejemplo))

    with mlflow.start_run(run_name=nombre_corrida) as corrida:
        mlflow.set_tags({
            "fase": "candidato",
            "familia": familia,
            "git_commit": commit,
            "git_cambios_sin_commit": str(cambios_pendientes).lower(),
        })

        mlflow.log_params({
            **datos_corrida,
            **{f"modelo__{nombre}": valor for nombre, valor in parametros.items()},
            "sha256_dataset_desarrollo": hash_dataset,
            "modelo": type(estimador).__name__,
            "semilla": parametros.get("random_state", "no_aplica"),
            "n_pliegues": cv_temporal.n_splits,
            "gap": cv_temporal.gap,
            "inicio_holdout": inicio_prueba.date().isoformat(),
            "p80_desarrollo": p80_desarrollo,
        })

        mlflow.log_metrics({
            "roc_auc_train_media_cv": float(resultados["roc_auc_train"].mean()),
            "roc_auc_val_media_cv": float(resultados["roc_auc_val"].mean()),
            "roc_auc_val_std_cv": float(resultados["roc_auc_val"].std(ddof=1)),
            "brecha_roc_auc_cv": float(resultados["brecha_roc_auc"].mean()),
            "segundos_ajuste_cv_total": float(resultados["segundos_ajuste"].sum()),
            "segundos_reajuste_desarrollo": segundos_reajuste,
        })

        mlflow.log_text(resultados.to_csv(index=False), "resultados_por_pliegue.csv")
        modelo = mlflow.sklearn.log_model(
            pipeline,
            name=nombre_pipeline,
            signature=firma,
            input_example=ejemplo,
            pyfunc_predict_fn="predict_proba",
        )

    print("Run ID:", corrida.info.run_id)
    print("URI del modelo:", modelo.model_uri)


registrar_candidato(
    bosque,
    resultados_bosque,
    nombre_corrida="candidato_random_forest_default",
    familia="bagging",
    nombre_pipeline="pipeline_random_forest",
)

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/22 16:19:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run candidato_random_forest_default at: http://127.0.0.1:5000/#/experiments/1/runs/88a42b72b1d74abeb5e513798f264ef0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: 88a42b72b1d74abeb5e513798f264ef0
URI del modelo: models:/m-4865249a6fe3465caa3142d697638910


## Tercer Candidato: HistGradientBoosting

In [17]:
from sklearn.ensemble import HistGradientBoostingClassifier

boosting = HistGradientBoostingClassifier(random_state=SEMILLA)
resultados_boosting = evaluar_candidato(boosting)

display(resultados_boosting.round(4))

auc = resultados_boosting["roc_auc_val"]
print(f"ROC-AUC CV: {auc.mean():.4f} ± {auc.std(ddof=1):.4f}")
print(f"Brecha media: {resultados_boosting['brecha_roc_auc'].mean():.4f}")
print("Learning rate:", boosting.get_params()["learning_rate"])

,pliegue,fin_train,inicio_validacion,fin_validacion,p80_train,roc_auc_train,roc_auc_val,brecha_roc_auc,segundos_ajuste
0,1,2012-11-05,2012-11-13,2014-12-29,1.8833,1.0,0.6390,0.3610,237.3972
1,2,2014-12-19,2014-12-30,2017-02-13,1.8898,1.0,0.6530,0.3470,1.5598
2,3,2017-02-06,2017-02-14,2019-04-01,1.8949,1.0,0.7341,0.2659,1.3163
3,4,2019-03-25,2019-04-02,2021-05-14,1.8921,1.0,0.6137,0.3863,1.8874
4,5,2021-05-07,2021-05-17,2023-06-30,2.0642,1.0,0.6663,0.3337,1.8118


ROC-AUC CV: 0.6612 ± 0.0452
Brecha media: 0.3388
Learning rate: 0.1


In [18]:
registrar_candidato(
    boosting,
    resultados_boosting,
    nombre_corrida="candidato_hist_gradient_boosting_default",
    familia="boosting",
    nombre_pipeline="pipeline_hist_gradient_boosting",
)

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/22 16:30:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which c

🏃 View run candidato_hist_gradient_boosting_default at: http://127.0.0.1:5000/#/experiments/1/runs/353404c1476247d0b3fb033570a5aa51
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: 353404c1476247d0b3fb033570a5aa51
URI del modelo: models:/m-77e4fde11b9d42868096e08f2c843570


## Cuarto Candidato: Multi Layer Perceptron

In [19]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(random_state=SEMILLA)
resultados_mlp = evaluar_candidato(mlp)

display(resultados_mlp.round(4))

auc = resultados_mlp["roc_auc_val"]
print(f"ROC-AUC CV: {auc.mean():.4f} ± {auc.std(ddof=1):.4f}")
print(f"Brecha media: {resultados_mlp['brecha_roc_auc'].mean():.4f}")
print("Learning rate inicial:", mlp.get_params()["learning_rate_init"])

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and 

,pliegue,fin_train,inicio_validacion,fin_validacion,p80_train,roc_auc_train,roc_auc_val,brecha_roc_auc,segundos_ajuste
0,1,2012-11-05,2012-11-13,2014-12-29,1.8833,0.9901,0.6082,0.3819,6.0504
1,2,2014-12-19,2014-12-30,2017-02-13,1.8898,0.9837,0.6701,0.3136,2.1208
2,3,2017-02-06,2017-02-14,2019-04-01,1.8949,0.9055,0.7424,0.1631,2.4634
3,4,2019-03-25,2019-04-02,2021-05-14,1.8921,0.9710,0.5378,0.4332,2.8821
4,5,2021-05-07,2021-05-17,2023-06-30,2.0642,0.9761,0.6216,0.3544,7.4078


ROC-AUC CV: 0.6360 ± 0.0760
Brecha media: 0.3292
Learning rate inicial: 0.001


In [20]:
registrar_candidato(
    mlp,
    resultados_mlp,
    nombre_corrida="candidato_mlp_default",
    familia="red_neuronal",
    nombre_pipeline="pipeline_mlp",
)

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-mi

🏃 View run candidato_mlp_default at: http://127.0.0.1:5000/#/experiments/1/runs/fb70dc2201724331be14e9817aed37f0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Run ID: fb70dc2201724331be14e9817aed37f0
URI del modelo: models:/m-e80d6fb2a9e5479c9c4b36025a5f6736


## Quinto Candidato: Chronos

In [ ]:
# from importlib.metadata import version
# from time import perf_counter

# print("1. Importando PyTorch...", flush=True)
# import torch

# print("2. Importando Chronos...", flush=True)
# from chronos import ChronosPipeline

# MODELO_CHRONOS = "amazon/chronos-t5-small"
# CONTEXTO_CHRONOS = 512
# MUESTRAS_CHRONOS = 20

# SEMILLA = 42

# torch.manual_seed(SEMILLA)

# print("3. Cargando pesos de Chronos en CPU...", flush=True)
# inicio = perf_counter()

# chronos = ChronosPipeline.from_pretrained(
#     MODELO_CHRONOS,
#     device_map="cpu",
# )

# print(f"Modelo cargado en {perf_counter() - inicio:.1f} s", flush=True)
# print("chronos-forecasting:", version("chronos-forecasting"))
# print("PyTorch:", torch.__version__)

1. Importando PyTorch...
2. Importando Chronos...


KeyboardInterrupt: 

## Quinto Candidato: SVM

In [7]:
from time import perf_counter

from sklearn.base import clone
from sklearn.metrics import roc_auc_score

from src.pipeline import crear_pipeline


def evaluar_candidato(estimador):
    resultados = []

    for pliegue, (idx_train, idx_val) in enumerate(
        cv_temporal.split(X_desarrollo), start=1
    ):
        X_train = X_desarrollo.iloc[idx_train]
        X_val = X_desarrollo.iloc[idx_val]
        vol_train = vol_desarrollo.iloc[idx_train]
        vol_val = vol_desarrollo.iloc[idx_val]

        p80 = vol_train.quantile(0.80)
        y_train = vol_train.gt(p80).astype(int)
        y_val = vol_val.gt(p80).astype(int)

        pipeline = crear_pipeline(clone(estimador), columnas_numericas)

        inicio = perf_counter()
        pipeline.fit(X_train, y_train)
        segundos_ajuste = perf_counter() - inicio

        auc_train = roc_auc_score(y_train, pipeline.predict_proba(X_train)[:, 1])
        auc_val = roc_auc_score(y_val, pipeline.predict_proba(X_val)[:, 1])

        resultados.append({
            "pliegue": pliegue,
            "fin_train": X_train.index[-1].date(),
            "inicio_validacion": X_val.index[0].date(),
            "fin_validacion": X_val.index[-1].date(),
            "p80_train": p80,
            "roc_auc_train": auc_train,
            "roc_auc_val": auc_val,
            "brecha_roc_auc": auc_train - auc_val,
            "segundos_ajuste": segundos_ajuste,
        })

    return pd.DataFrame(resultados)

In [8]:
from sklearn.svm import SVC

SEMILLA = 42

svm = SVC(probability=True, random_state=SEMILLA)
resultados_svm = evaluar_candidato(svm)

display(resultados_svm.round(4))

auc = resultados_svm["roc_auc_val"]
print(f"ROC-AUC CV: {auc.mean():.4f} ± {auc.std(ddof=1):.4f}")
print(f"Brecha media: {resultados_svm['brecha_roc_auc'].mean():.4f}")

,pliegue,fin_train,inicio_validacion,fin_validacion,p80_train,roc_auc_train,roc_auc_val,brecha_roc_auc,segundos_ajuste
0,1,2012-11-05,2012-11-13,2014-12-29,1.8833,0.9645,0.6309,0.3337,0.3318
1,2,2014-12-19,2014-12-30,2017-02-13,1.8898,0.9430,0.7019,0.2411,0.4288
2,3,2017-02-06,2017-02-14,2019-04-01,1.8949,0.9297,0.7347,0.1950,0.9239
3,4,2019-03-25,2019-04-02,2021-05-14,1.8921,0.9106,0.7135,0.1971,1.8178
4,5,2021-05-07,2021-05-17,2023-06-30,2.0642,0.9089,0.7165,0.1925,3.2242


ROC-AUC CV: 0.6995 ± 0.0401
Brecha media: 0.2319


In [ ]:
import subprocess
from hashlib import sha256
from time import perf_counter

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.base import clone

from src.pipeline import crear_pipeline


def registrar_candidato(
    estimador, resultados, *, nombre_corrida, familia, nombre_pipeline
):
    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=raiz, text=True
    ).strip()

    cambios_pendientes = bool(subprocess.check_output(
        ["git", "status", "--porcelain", "-uno"], cwd=raiz, text=True
    ).strip())

    hash_dataset = sha256(
        entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
    ).hexdigest()

    p80_desarrollo = float(vol_desarrollo.quantile(0.80))
    y_desarrollo = vol_desarrollo.gt(p80_desarrollo).astype(int)

    pipeline = crear_pipeline(clone(estimador), columnas_numericas)
    inicio = perf_counter()
    pipeline.fit(X_desarrollo, y_desarrollo)
    segundos_reajuste = perf_counter() - inicio

    parametros = estimador.get_params(deep=False)
    ejemplo = X_desarrollo.head(5).copy()
    firma = infer_signature(ejemplo, pipeline.predict_proba(ejemplo))

    with mlflow.start_run(run_name=nombre_corrida) as corrida:
        mlflow.set_tags({
            "fase": "candidato",
            "familia": familia,
            "git_commit": commit,
            "git_cambios_sin_commit": str(cambios_pendientes).lower(),
        })

        mlflow.log_params({
            **datos_corrida,
            **{f"modelo__{nombre}": valor for nombre, valor in parametros.items()},
            "sha256_dataset_desarrollo": hash_dataset,
            "modelo": type(estimador).__name__,
            "semilla": parametros.get("random_state", "no_aplica"),
            "n_pliegues": cv_temporal.n_splits,
            "gap": cv_temporal.gap,
            "inicio_holdout": inicio_prueba.date().isoformat(),
            "p80_desarrollo": p80_desarrollo,
        })

        mlflow.log_metrics({
            "roc_auc_train_media_cv": float(resultados["roc_auc_train"].mean()),
            "roc_auc_val_media_cv": float(resultados["roc_auc_val"].mean()),
            "roc_auc_val_std_cv": float(resultados["roc_auc_val"].std(ddof=1)),
            "brecha_roc_auc_cv": float(resultados["brecha_roc_auc"].mean()),
            "segundos_ajuste_cv_total": float(resultados["segundos_ajuste"].sum()),
            "segundos_reajuste_desarrollo": segundos_reajuste,
        })

        mlflow.log_text(
            resultados.to_csv(index=False),
            "resultados_por_pliegue.csv",
        )

        modelo = mlflow.sklearn.log_model(
            pipeline,
            name=nombre_pipeline,
            signature=firma,
            input_example=ejemplo,
            pyfunc_predict_fn="predict_proba",
        )

    print("Run ID:", corrida.info.run_id)
    print("URI del modelo:", modelo.model_uri)


registrar_candidato(
    svm,
    resultados_svm,
    nombre_corrida="candidato_svm",
    familia="maquina_vectores_soporte",
    nombre_pipeline="pipeline_svm",
)

NameError: name 'registrar_candidato' is not defined